## Assignment Machine Learning: Ela, Emma, Florence, Jelle -- 28-03-2026

Importing packages 

In [36]:
import pandas as pd
import os
import zipfile

import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns 

from sklearn import datasets as ds
from os import name
from scipy.stats import loguniform, randint 
from sklearn.model_selection import train_test_split, GridSearchCV, StratifiedKFold
from sklearn.decomposition import PCA
from sklearn import metrics
from sklearn import model_selection
from sklearn import preprocessing
from sklearn import decomposition

from sklearn.naive_bayes import GaussianNB
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.linear_model import LogisticRegression
from sklearn.linear_model import SGDClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn import neighbors

from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import OneHotEncoder, StandardScaler, RobustScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
import sklearn.metrics as sklm
from sklearn.ensemble import RandomForestClassifier
from sklearn.cross_decomposition import PLSRegression
from sklearn.pipeline import FunctionTransformer
from sklearn.impute import SimpleImputer
from sklearn.neural_network import MLPClassifier
from sklearn.decomposition import PCA
from sklearn.svm import SVC
from sklearn.tree import plot_tree
from sklearn.metrics import precision_score, recall_score, f1_score, roc_curve, auc, confusion_matrix, classification_report, average_precision_score, roc_auc_score

from scipy.stats import ttest_ind

from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import RobustScaler
from scipy.stats import randint
from sklearn.model_selection import RandomizedSearchCV

scores_dict = {}


Load data 

In [21]:
with zipfile.ZipFile("ecg_data.zip","r") as zip_ref:
    zip_ref.extractall("ecg_data")

def load_data():
    this_directory = os.getcwd()
    data = pd.read_csv(os.path.join(this_directory, 'ecg_data/ecg_data.csv'), index_col=0)
    return data

raw_data = load_data()

Data description

In [22]:
print(f'The number of samples: {len(raw_data.index)}')
print(f'The number of columns: {len(raw_data.columns)}')

print(f'The number of NaN values in the entire dataframe: {raw_data.isnull().sum().sum()}')
print(f'The number of samples with label 0: {len(raw_data[raw_data["label"] == 0])}')
print(f'The number of samples with label 1: {len(raw_data[raw_data["label"] == 1])}')
print(f'The percentage of samples with label 0 is thus {len(raw_data[raw_data["label"] == 0])/len(raw_data.index)*100:.2f}%', 
      f'and the percentage with label 1 {len(raw_data[raw_data["label"] == 1])/len(raw_data.index)*100:.2f}%')


The number of samples: 827
The number of columns: 9001
The number of NaN values in the entire dataframe: 0
The number of samples with label 0: 681
The number of samples with label 1: 146
The percentage of samples with label 0 is thus 82.35% and the percentage with label 1 17.65%


Splitting the data in a training and test set

In [23]:
X = raw_data.drop('label', axis=1)
Y = raw_data['label']

x_train, x_test, y_train, y_test = train_test_split(X, Y, test_size=0.20, random_state=4, stratify=Y)

Preprocessing of data

In [ ]:
class VarianceFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.01):
        self.threshold = threshold

    def fit(self, X, y=None):
        self.columns_to_drop_ = [col for col in X.columns if np.var(X[col]) < self.threshold]
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


class CorrelationFilter(BaseEstimator, TransformerMixin):
    def __init__(self, threshold=0.95):
        self.threshold = threshold

    def fit(self, X, y=None):
        corr_matrix = X.corr().abs()
        upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
        self.columns_to_drop_ = [col for col in upper.columns if any(upper[col] > self.threshold)]
        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


class TTestFilter(BaseEstimator, TransformerMixin):
    def __init__(self, alpha=0.05):
        self.alpha = alpha

    def fit(self, X, y):
        if isinstance(y, pd.DataFrame):
            y = y.iloc[:, 0]

        self.columns_to_drop_ = []
        threshold = self.alpha / X.shape[1]  # Bonferroni

        for col in X.columns:
            group0 = X.loc[y == 0, col]
            group1 = X.loc[y == 1, col]
            _, p_value = ttest_ind(group0, group1, nan_policy="omit")

            if p_value > threshold:
                self.columns_to_drop_.append(col)

        return self

    def transform(self, X):
        return X.drop(columns=self.columns_to_drop_, errors="ignore")


pipeline = Pipeline([
    ("variance", VarianceFilter(threshold=0.01)),
    ("correlation", CorrelationFilter(threshold=0.95)),
    ("ttest", TTestFilter(alpha=0.05)),
    ("scaler", RobustScaler())
    ])

pipeline.fit(x_train, y_train)
x_train_preprocessed = pipeline.transform(x_train)
x_test_preprocessed = pipeline.transform(x_test)

#amount of features after preprocessing 
x_train_var = pipeline.named_steps["variance"].transform(x_train)
x_train_corr = pipeline.named_steps["correlation"].transform(x_train_var)
x_train_t = pipeline.named_steps["ttest"].transform(x_train_corr)

print(f"Start amount features: {x_train.shape[1]}")
print(f"After the variance filter: {x_train_var.shape[1]}")
print(f"After the correlation filter: {x_train_corr.shape[1]}")
print(f"After the t-test filter: {x_train_t.shape[1]}")

Start amount features: 9000
After the variance filter: 9000
After the correlation filter: 4068
After the t-test filter: 47


Function for the evaluation metrics on test set (Precision, Recall, F1, AUC-PR, ROC-AUC and plot of AUC-PR)

In [ ]:
def evaluate_model_on_test(clf, x_test, y_test, name="Model"):
    '''
        Call function as follows:
        - scores_dict.update(evaluate_model_on_test(...))

        Naming convention for models:
        - LR        -->     Lasso Logistic Regression
        - RF        -->     Random Forest
        - SVM       -->     Support Vector Machine
        - XGB       -->     XGBoost
        - PLS-DA    -->     Partial Least-Squares Discriminant Analysis
        - NN        -->     Neural Network
        
    '''
    y_pred = clf.predict(x_test)
    if hasattr(clf, 'predict_proba'):
        y_score = clf.predict_proba(x_test)[:, 1]
    else:
        y_score = clf.decision_function(x_test)

    # Calculate metrics
    precision = precision_score(y_test, y_pred, zero_division=1)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    auc_pr = average_precision_score(y_test, y_score)
    auc_roc = roc_auc_score(y_test, y_score)

    # Print results
    print(f"--- Evaluation: {name} ---")
    print(f"Precision:  {precision:.4f}")
    print(f"Recall:     {recall:.4f}")
    print(f"F1-Score:   {f1:.4f}")
    print(f"AUC-PR:     {auc_pr:.4f}")
    print(f"ROC-AUC:    {auc_roc:.4f}")
    print("-" * 30)

    return {name: y_score}


Logistic regression model

In [29]:
kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=4)

pipeline_lr = Pipeline([
    ("variance", VarianceFilter(threshold=0.01)),
    ("correlation", CorrelationFilter(threshold=0.95)),
    ("ttest", TTestFilter(alpha=0.05)),
    ('scaler', RobustScaler()),  #data scaling is important for model convergence with the saga solver
    ('classifier', LogisticRegression(
        solver='saga',
        random_state=4,
        max_iter=5000  # important for convergence
    ))
])

param_dist = {
    'classifier__C': loguniform(1e-3, 10),  
    'classifier__penalty': ['l1', 'l2'],
    'classifier__class_weight': [None, 'balanced']
}

random_search = RandomizedSearchCV(
    pipeline_lr,
    param_distributions=param_dist,
    n_iter=50,  # number of random combinations
    cv=kf,
    scoring='average_precision',
    n_jobs=-1,
    random_state=4
)

random_search.fit(x_train, y_train)

print('Best parameters found:\n', random_search.best_params_)
print("Best score:", random_search.best_score_)

c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best parameters found:
 {'classifier__C': np.float64(0.05553753341120423), 'classifier__class_weight': None, 'classifier__penalty': 'l1'}
Best score: 0.5352283241363913


In [30]:
param_grid_lr = {
    'classifier__C': [0.01, 0.05, 0.1, 0.15],
    'classifier__penalty': ['l1'],  
    'classifier__class_weight': [None],  
}

grid_search_lr = GridSearchCV(pipeline_lr, param_grid_lr, cv=kf, scoring='average_precision', n_jobs=-1)
grid_search_lr.fit(x_train, y_train)

print('Best parameters found:\n', grid_search_lr.best_params_)
print("Beste score:", grid_search_lr.best_score_)

c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(


Best parameters found:
 {'classifier__C': 0.05, 'classifier__class_weight': None, 'classifier__penalty': 'l1'}
Beste score: 0.5366527414450639


In [31]:
lr_model = grid_search_lr.best_estimator_ 

evaluate_model_on_test(lr_model, x_test, y_test, name="LR")

--- Evaluation: LR ---
Precision:  0.6667
Recall:     0.1379
F1-Score:   0.2286
AUC-PR:     0.4066
ROC-AUC:    0.7063
------------------------------


{'LR': array([0.16775188, 0.10906859, 0.20949933, 0.02473604, 0.11094328,
        0.10284051, 0.14272396, 0.16192724, 0.23427984, 0.17197977,
        0.484231  , 0.04171173, 0.19556945, 0.05008294, 0.15519542,
        0.07442404, 0.20447696, 0.08628423, 0.12787841, 0.26211204,
        0.15400081, 0.22882699, 0.23416774, 0.10289795, 0.07888164,
        0.2122582 , 0.08736859, 0.18114893, 0.18554714, 0.50250864,
        0.26230485, 0.13149865, 0.11284249, 0.04116932, 0.10645621,
        0.12748659, 0.10113632, 0.12054587, 0.14980309, 0.16001081,
        0.11857057, 0.06049937, 0.06356935, 0.016796  , 0.19324059,
        0.08400462, 0.11098586, 0.1375807 , 0.51030515, 0.3900295 ,
        0.03812281, 0.05764771, 0.41712832, 0.21338746, 0.14882655,
        0.14850917, 0.03371119, 0.079248  , 0.11719375, 0.08691854,
        0.20892716, 0.10153341, 0.10847116, 0.10143833, 0.70496743,
        0.993655  , 0.15268725, 0.18446422, 0.12347693, 0.23218591,
        0.08441335, 0.09482759, 0.21302762

Performing Random forest: 

In [ ]:
# y_train en y_test series van maken
if isinstance(y_train, pd.DataFrame):
    if y_train.shape[1] == 1:
        y_train = y_train.iloc[:, 0]
    else:
        raise ValueError("y_train moet 1 kolom bevatten.")

if isinstance(y_test, pd.DataFrame):
    if y_test.shape[1] == 1:
        y_test = y_test.iloc[:, 0]
    else:
        raise ValueError("y_test moet 1 kolom bevatten.")

pipeline_rf = Pipeline([
    ("variance", VarianceFilter(threshold=0.01)),
    ("correlation", CorrelationFilter(threshold=0.95)),
    ("ttest", TTestFilter(alpha=0.05)),
    ("scaler", RobustScaler()),
    ("rf", RandomForestClassifier(random_state=4, class_weight="balanced"))
])


param_dist = {
    "rf__n_estimators": randint(100, 600),
    "rf__max_depth": [None, 5, 10, 20, 30, 50],
    "rf__min_samples_split": randint(2, 20),
    "rf__min_samples_leaf": randint(1, 10),
    "rf__max_features": ["sqrt", "log2", None],
    "rf__bootstrap": [True, False]
}

kf = StratifiedKFold(n_splits=5, shuffle=True, random_state=4)
random_search = RandomizedSearchCV(
    estimator=pipeline_rf,
    param_distributions=param_dist,
    n_iter=30,
    scoring="average_precision",
    cv=kf,
    verbose=0,
    n_jobs=1,
    random_state=4,
    return_train_score=True
)

random_search.fit(x_train, y_train)

print("Beste parameters:")
print(random_search.best_params_)
print(f"\nBeste cross-validated PR-AUC: {random_search.best_score_:.4f}")

best_rf = random_search.best_estimator_

evaluate_model_on_test(
    best_rf,
    x_test,
    y_test,
    name="RF"
)

Beste parameters:
{'rf__bootstrap': True, 'rf__max_depth': 20, 'rf__max_features': 'sqrt', 'rf__min_samples_leaf': 8, 'rf__min_samples_split': 8, 'rf__n_estimators': 140}

Beste cross-validated PR-AUC: 0.5702
--- Evaluation: Optimized Random Forest ---
Precision:  0.4167
Recall:     0.3448
F1-Score:   0.3774
AUC-PR:     0.4932
ROC-AUC:    0.7727
------------------------------


{'Optimized Random Forest': array([0.32274403, 0.34058256, 0.32479916, 0.15158401, 0.17398736,
        0.3104582 , 0.35872866, 0.21471847, 0.36034488, 0.25382521,
        0.38533013, 0.04584769, 0.3989813 , 0.09420022, 0.30005858,
        0.10528347, 0.40692298, 0.08930127, 0.12432594, 0.61313915,
        0.30703313, 0.29129572, 0.60923817, 0.16142967, 0.23046264,
        0.19628822, 0.02594157, 0.3049576 , 0.27199155, 0.61269555,
        0.51328908, 0.18608531, 0.20589663, 0.11563813, 0.09961384,
        0.28309812, 0.09580514, 0.30496093, 0.31552133, 0.34623642,
        0.13634808, 0.29696176, 0.09607697, 0.25117524, 0.43238702,
        0.13179344, 0.17923065, 0.1912782 , 0.78056672, 0.62663142,
        0.13127182, 0.08018656, 0.52920297, 0.38535742, 0.29598789,
        0.23374964, 0.20454415, 0.51449818, 0.15735077, 0.13966168,
        0.31355456, 0.11731636, 0.14453304, 0.08980641, 0.9478581 ,
        0.95956034, 0.32741572, 0.4784938 , 0.13068879, 0.2882663 ,
        0.08500622, 0

Partial Least-Squares Discriminant Analysis (PLS-DA)

In [37]:
def squeeze_output(X):
    if isinstance(X, tuple):
        X = X[0]       
    return X.reshape(X.shape[0], -1)

pipeline_pls_da = Pipeline([
    ("variance", VarianceFilter(threshold=0.01)),
    ("correlation", CorrelationFilter(threshold=0.95)),
    ("ttest", TTestFilter(alpha=0.05)),
    ('scaler', RobustScaler()),
    ('pls', PLSRegression(scale=False, max_iter=10)),
    ('squeeze', FunctionTransformer(squeeze_output)),
    ('classifier', LogisticRegression(solver='saga', random_state=4, max_iter=10))
    ])

param_dist = {
    'pls__n_components': randint(1, 25),
    'classifier__C': loguniform(1e-3, 10),  
    'classifier__penalty': ['l1', 'l2'],
    'classifier__class_weight': [None, 'balanced']
}

random_search = RandomizedSearchCV(
    pipeline_pls_da,
    param_distributions=param_dist,
    n_iter=50,  # number of random combinations
    cv=kf,
    scoring='average_precision',
    n_jobs=-1,
    random_state=4
)

random_search.fit(x_train, y_train)

print('Best parameters found:\n', random_search.best_params_)
print("Best score:", random_search.best_score_)


Best parameters found:
 {'classifier__C': np.float64(0.037249864670657974), 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1', 'pls__n_components': 3}
Best score: 0.5402856481006166


c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [38]:
param_grid_pls_da = {
    'pls__n_components': [2, 3, 5],
    'classifier__C': [0.01, 0.05, 0.1],
    'classifier__penalty': ['l1'],
    'classifier__class_weight': ['balanced']
    }

grid_search_pls_da = GridSearchCV(pipeline_pls_da, param_grid_pls_da, cv=kf, scoring="average_precision", n_jobs=-1)
grid_search_pls_da.fit(x_train, y_train)

print('Best parameters found:\n', grid_search_pls_da.best_params_)
print("Beste score:", grid_search_pls_da.best_score_)

Best parameters found:
 {'classifier__C': 0.05, 'classifier__class_weight': 'balanced', 'classifier__penalty': 'l1', 'pls__n_components': 3}
Beste score: 0.537312417579171


c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1135: FutureWarning: 'penalty' was deprecated in version 1.8 and will be removed in 1.10. To avoid this warning, leave 'penalty' set to its default value and use 'l1_ratio' or 'C' instead. Use l1_ratio=0 instead of penalty='l2', l1_ratio=1 instead of penalty='l1', and C=np.inf instead of penalty=None.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_logistic.py:1160: UserWarning: Inconsistent values: penalty=l1 with l1_ratio=0.0. penalty is deprecated. Please use l1_ratio only.
  warnings.warn(
c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\linear_model\_sag.py:348: ConvergenceWarning: The max_iter was reached which means the coef_ did not converge
  warnings.warn(


In [39]:
pls_da_model = grid_search_pls_da.best_estimator_ 

evaluate_model_on_test(pls_da_model, x_test, y_test, name="PLS-DA")

--- Evaluation: PLS-DA ---
Precision:  0.3220
Recall:     0.6552
F1-Score:   0.4318
AUC-PR:     0.3773
ROC-AUC:    0.7100
------------------------------


{'PLS-DA': array([0.56105781, 0.40544889, 0.427512  , 0.10545894, 0.52721539,
        0.42260334, 0.39661956, 0.4703386 , 0.56857389, 0.52046773,
        0.9378229 , 0.13352768, 0.55921958, 0.23800664, 0.47487055,
        0.27382045, 0.53844729, 0.2808949 , 0.35368925, 0.67352805,
        0.29538845, 0.65394823, 0.65279354, 0.40849337, 0.415048  ,
        0.57252656, 0.24326131, 0.52620101, 0.46382656, 0.91791878,
        0.70720061, 0.46398049, 0.30624097, 0.16432011, 0.42723747,
        0.34917102, 0.29752169, 0.36864636, 0.43137424, 0.42590617,
        0.38278286, 0.31055425, 0.25518064, 0.07282163, 0.59167807,
        0.21145068, 0.28745756, 0.47768452, 0.90919888, 0.92696392,
        0.26234173, 0.21543487, 0.80140012, 0.55482007, 0.27081512,
        0.42295368, 0.19721825, 0.29900465, 0.43564372, 0.40662357,
        0.51963448, 0.33475375, 0.32212223, 0.24189233, 0.97223429,
        0.99912566, 0.51413433, 0.51778291, 0.36084466, 0.69781398,
        0.32258919, 0.39051003, 0.6853

Feed-forward neural network

In [33]:
pipeline_NN = Pipeline(steps=[
    ("variance", VarianceFilter(threshold=0.01)),
    ("correlation", CorrelationFilter(threshold=0.95)),
    ("ttest", TTestFilter(alpha=0.05)),
    ('scaler', RobustScaler()),
    ('classifier', MLPClassifier(early_stopping=True, random_state=4)) 
])

param_grid_NN = {
    'classifier__hidden_layer_sizes': [(32, 16, 8), (64,), (128, 64)],
    'classifier__activation': ['relu', 'tanh'],
    'classifier__solver': ['adam', 'sgd', 'lbfgs'],  
    'classifier__learning_rate_init': loguniform(1e-4, 1e-2),
    'classifier__alpha': loguniform(1e-5, 1e-2),
    'classifier__max_iter': randint(150, 500),
}

random_search_NN = RandomizedSearchCV(
    pipeline_NN,
    param_distributions=param_grid_NN,
    n_iter=50,  # number of random combinations
    cv=kf,
    scoring='average_precision',
    n_jobs=-1,
    random_state=4
)

random_search_NN.fit(x_train, y_train)

print('Best parameters found:\n', random_search_NN.best_params_)
print("Best score:", random_search_NN.best_score_)

Best parameters found:
 {'classifier__activation': 'tanh', 'classifier__alpha': np.float64(9.851979721981269e-05), 'classifier__hidden_layer_sizes': (128, 64), 'classifier__learning_rate_init': np.float64(0.004936788795482597), 'classifier__max_iter': 281, 'classifier__solver': 'adam'}
Best score: 0.5202518279121435


In [34]:
param_grid_NN = {
    'classifier__hidden_layer_sizes': [(65,), (128, 64)],
    'classifier__activation': ['tanh', 'relu'],
    'classifier__solver': ['adam','lbfgs'],
    'classifier__learning_rate_init': [1e-2, 1e-3, 5e-3],
    'classifier__alpha': [1e-4, 5e-5, 5e-4],
    'classifier__learning_rate': ['constant'],
    'classifier__max_iter': [250, 275, 300]
}

grid_search_NN = GridSearchCV(pipeline_NN, param_grid_NN, cv=kf, scoring='average_precision', n_jobs=-1)
grid_search_NN.fit(x_train, y_train)

print('Best parameters found:\n', grid_search_NN.best_params_)
print("Best score:", grid_search_NN.best_score_)

c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\model_selection\_validation.py:490: FitFailedWarning: 
1 fits failed out of a total of 1080.
The score on these train-test partitions for these parameters will be set to nan.
If these failures are not expected, you can try to debug them by setting error_score='raise'.

Below are more details about the failures:
--------------------------------------------------------------------------------
1 fits failed with the following error:
Traceback (most recent call last):
  File "c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\model_selection\_validation.py", line 833, in _fit_and_score
    estimator.fit(X_train, y_train, **fit_params)
  File "c:\Users\emma_\miniconda3\envs\technical_medicine\Lib\site-packages\sklearn\base.py", line 1336, in wrapper
    return fit_method(estimator, *args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\emma_\miniconda3\envs\techn

Best parameters found:
 {'classifier__activation': 'tanh', 'classifier__alpha': 0.0001, 'classifier__hidden_layer_sizes': (128, 64), 'classifier__learning_rate': 'constant', 'classifier__learning_rate_init': 0.005, 'classifier__max_iter': 250, 'classifier__solver': 'adam'}
Best score: 0.5077678197067753


In [ ]:
NN_model = grid_search_NN.best_estimator_ 


evaluate_model_on_test(NN_model, x_test, y_test, name="NN")

--- Evaluation: NN ---
Precision:  0.5238
Recall:     0.3793
F1-Score:   0.4400
AUC-PR:     0.4216
ROC-AUC:    0.7402
------------------------------


{'NN': array([1.06631866e-01, 2.46948593e-01, 2.01801252e-01, 2.54792014e-03,
        6.14136570e-02, 8.61189442e-02, 8.25196399e-02, 2.99868633e-02,
        8.35179272e-01, 3.16160680e-01, 7.75767780e-01, 1.51455413e-03,
        4.86108126e-01, 3.73857841e-03, 5.05044407e-02, 2.54250151e-02,
        4.42465024e-01, 4.28664050e-03, 3.05304682e-02, 3.95806189e-01,
        5.02091734e-03, 5.01174328e-01, 1.65733923e-01, 9.47958350e-03,
        1.44660999e-02, 1.26426193e-01, 3.73857330e-03, 6.97219245e-02,
        6.30111876e-02, 8.89044193e-01, 8.53885557e-02, 1.63813002e-02,
        5.14678280e-03, 1.26646909e-03, 9.07809777e-03, 8.51711228e-02,
        5.93394325e-03, 1.18949668e-02, 1.20398935e-01, 7.66981696e-02,
        4.82912950e-03, 1.50403274e-01, 1.53196768e-03, 1.55503710e-03,
        2.83981689e-01, 4.89551285e-03, 3.42882297e-03, 1.34997121e-02,
        3.88667709e-01, 2.15623563e-01, 9.04374238e-03, 1.79831538e-03,
        2.37122585e-01, 6.37264311e-01, 3.56815962e-02, 9.